# GPU Backtest on Colab

Run PatchTSMixer or AE+Ridge GPU backtests on Google Colab.

**Requirements:** Colab GPU runtime (T4 or better). Go to *Runtime > Change runtime type > GPU*.

In [ ]:
# --- 1. Setup: clone repo and install deps ---
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"  # FILL IN: your repo URL
REPO_DIR = "harxhar"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrow

In [ ]:
# --- 2. Verify GPU ---
import torch

assert torch.cuda.is_available(), "No GPU detected! Change runtime to GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- 3. Configure CUDA ---
import os

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

In [ ]:
# --- 4. Upload your data ---
# Option A: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # upload your parquet files to all30min/

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/harxhar_data/all30min'

# Option C: Data already in repo
DATA_PATH = "all30min"

## PatchTSMixer Backtest

In [ ]:
import copy

from src.backtest.gpu_engine import run_multigpu_backtest
from src.core.config import DL_CONFIG
from src.data import load_and_prep_data_strided
from src.evaluation.metrics import calculate_global_metrics

config = copy.deepcopy(DL_CONFIG)
config["data_path"] = DATA_PATH
config["gpu_count"] = 1  # Colab has 1 GPU

hparams = {
    "exog_cols": "none",
    "use_transform_exog": False,
    "use_diurnal": True,
    "allow_missing": False,
    "use_winsor": False,
}

X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    hparams, config["data_path"], lag=config["model"]["context_len"]
)
print(f"Data: {X_np.shape[0]:,} samples, {X_np.shape[1]} features")

In [ ]:
results = run_multigpu_backtest(
    X_np, y_np, dates, baselines, config,
    model_module="src.models.deep_learning",
)

metrics = calculate_global_metrics(results)
print(f"QLIKE: {metrics.get('qlike', float('nan')):.4f}")
print(f"MSE:   {metrics.get('mse', float('nan')):.6f}")
print(f"MAE:   {metrics.get('mae', float('nan')):.6f}")

## AE+Ridge Backtest

In [ ]:
import copy
import numpy as np

from src.backtest.gpu_engine_ae import run_ae_multigpu_backtest
from src.core.config import AE_RIDGE_GPU_CONFIG
from src.data import load_and_prep_data_strided
from src.evaluation.metrics import calculate_global_metrics

np.random.seed(42)

ae_config = copy.deepcopy(AE_RIDGE_GPU_CONFIG)
ae_config["data_path"] = DATA_PATH
ae_config["gpu_count"] = 1  # Colab has 1 GPU

ae_hparams = {
    "exog_cols": "none",
    "use_transform_exog": False,
    "use_diurnal": True,
    "allow_missing": False,
    "use_winsor": True,
    "feature_type": "raw",
    "lag_scope": "global",
}

X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    ae_hparams, ae_config["data_path"]
)
ae_config["model"]["n_features"] = X_np.shape[1]
print(f"Data: {X_np.shape[0]:,} samples, {X_np.shape[1]} features")

In [ ]:
ae_results = run_ae_multigpu_backtest(X_np, y_np, dates, baselines, ae_config)

ae_metrics = calculate_global_metrics(ae_results)
print(f"QLIKE: {ae_metrics.get('qlike', float('nan')):.4f}")
print(f"MSE:   {ae_metrics.get('mse', float('nan')):.6f}")
print(f"MAE:   {ae_metrics.get('mae', float('nan')):.6f}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

def plot_results(results, title="Volatility Forecast"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Time series (last 500 points)
    subset = results.iloc[-500:]
    axes[0].plot(subset.index, subset["true_raw"], label="True RV", alpha=0.6, color="black")
    axes[0].plot(subset.index, subset["pred_raw"], label="Predicted RV", alpha=0.8, color="#1f77b4")
    axes[0].set_title(title)
    axes[0].set_ylabel("Realized Volatility")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Scatter
    axes[1].scatter(results["true_raw"], results["pred_raw"], alpha=0.3, s=10)
    lims = [results["true_raw"].min(), results["true_raw"].max()]
    axes[1].plot(lims, lims, "k-", alpha=0.5)
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("True Volatility")
    axes[1].set_ylabel("Predicted Volatility")
    axes[1].set_title("Diagnostic Scatter")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Uncomment whichever you ran:
# plot_results(results, "PatchTSMixer")
# plot_results(ae_results, "AE+Ridge")

In [ ]:
# --- Download results ---
# from google.colab import files
# results.to_csv("patchts_results.csv", index=False)
# files.download("patchts_results.csv")